In [1]:
import pandas as pd
import json


In [2]:
# Load the categorize response JSONL file and extract custom_id and response
response_categorize = []
with open('../batch/responses_categorize_sample_20250930_124008.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        response_categorize.append({
            'id': data.get('custom_id'),
            'llm_category': data.get('response')
        })

response_df = pd.DataFrame(response_categorize)
response_df.head()


,id,llm_category
0,62160ea4-1d49-43ee-aa4b-7131a4ee6f64,Diagnosis
1,649e1475-8944-417a-b984-9067c9f203a2,Diagnosis
2,ca367e87-de96-4cb7-a097-6be2a041feb1,Procedure
3,c4fda87f-a779-4cba-9ecf-90190790a280,Procedure
4,33caf6f3-3820-4e80-a693-24c0830dfcea,Diagnosis


In [3]:
# Load the medmcqa dataset to get the questions
medmcqa_df = pd.read_csv('medmcqa.csv')
medmcqa_df.head()


,Unnamed: 0,id,question,opa,opb,opc,opd,cop,choice_type,exp,subject_name,topic_name
0,0,e9ad821a-c438-4965-9f77-760819dfa155,Chronic urethral obstruction due to benign pri...,Hyperplasia,Hyperophy,Atrophy,Dyplasia,2,single,Chronic urethral obstruction because of urinar...,Anatomy,Urinary tract
1,1,e3d3c4e1-4fb2-45e7-9f88-247cc8f373b3,Which vitamin is supplied from only animal sou...,Vitamin C,Vitamin B7,Vitamin B12,Vitamin D,2,single,Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. ...,Biochemistry,Vitamins and Minerals
2,2,5c38bea6-787a-44a9-b2df-88f4218ab914,All of the following are surgical options for ...,Adjustable gastric banding,Biliopancreatic diversion,Duodenal Switch,Roux en Y Duodenal By pass,3,multi,"Ans. is 'd' i.e., Roux en Y Duodenal Bypass Ba...",Surgery,Surgical Treatment Obesity
3,3,cdeedb04-fbe9-432c-937c-d53ac24475de,Following endaerectomy on the right common car...,Central aery of the retina,Infraorbital aery,Lacrimal aery,Nasociliary aretry,0,multi,The central aery of the retina is a branch of ...,Ophthalmology,NaN
4,4,dc6794a3-b108-47c5-8b1b-3b4931577249,Growth hormone has its effect on growth through?,Directly,IG1-1,Thyroxine,Intranuclear receptors,1,single,"Ans. is 'b' i.e., IGI-1GH has two major functi...",Physiology,NaN


In [4]:
# Merge the categorize responses with the questions data
df = pd.merge(medmcqa_df, response_df, on='id', how='inner')
print(f"Total merged records: {len(df)}")
df.head()


Total merged records: 387


,Unnamed: 0,id,question,opa,opb,opc,opd,cop,choice_type,exp,subject_name,topic_name,llm_category
0,458,4aed3cb7-f6b2-4bda-b2c0-1230c3e61e34,3-year-old child comes in ER with Hv'o vomitin...,No dehydration,Mild dehydration,Some dehydration,Severe dehydration,3,single,on A child with severe dehydration will have ...,Pediatrics,NaN,Diagnosis
1,2200,f31d9293-31b1-4fc3-9af1-70735f827e4c,"During the following procedure, optimum interv...",30 seconds,45 seconds,60 seconds,90 seconds,3,single,"Ans. D 90 secondsRef: Datta, 8th ed. pg. 673Th...",Physiology,"Heart, Circulation, and Blood",Procedure
2,2745,de396e96-6668-4cca-9bd7-e69620d07ee5,A pregnant woman with fibroid uterus develops ...,Preterm labour,Torsion of fibroid,Red degeneration of fibroid,Infection in fibroid,2,single,Red degeneration of fibroid: It results from s...,Gynaecology & Obstetrics,Anatomy of the female genital tract,Diagnosis
3,3008,69d9e480-742a-4692-9ef8-8cdf1d99da4e,76 year old man presents with lytic lesion in ...,Metastasis,Multiple myeloma,Osteomalacia,Hyperparathyroidism,1,single,"B i.e. Multiple Myeloma Old age, male sex & ve...",Radiology,NaN,Diagnosis
4,4015,5b98bc47-99f0-471a-9b86-894f77ab1681,A child is spinned around by holding his hand ...,Pulled elbow,Radial head dislocation,Annular ligament tear,Fracture olecranon process,0,multi,This is case of pulled elbow and a dramatic cu...,Orthopaedics,NaN,Diagnosis


In [5]:
# Show value counts for the LLM categories
print("Category distribution:")
category_counts = df['llm_category'].value_counts()
print(category_counts)
print(f"\nTotal questions: {category_counts.sum()}")


Category distribution:
llm_category
Diagnosis     303
Procedure      47
Medication     27
Other           7
Diagnosis.      2
Trivia          1
Name: count, dtype: int64

Total questions: 387


In [6]:
# Filter and show medication questions specifically
medication_questions = df[df['llm_category'] == 'Medication']
print(f"Number of medication questions: {len(medication_questions)}")
print("\nSample medication questions:")
for i, row in medication_questions.head(10).iterrows():
    print(f"\n{i+1}. {row['question']}")
    print(f"   Answer: {row['op' + 'abcd'[row['cop']]]}")
    print(f"   Subject: {row['subject_name']}")


Number of medication questions: 27

Sample medication questions:

8. This drug has activity against many strains of P. aeruginosa. However, when it is used alone, resistance has emerged during the course of treatment. The drug should not be used in penicillin-allergic patients. Its activity against gram-negative rods is enhanced if it is given in combination with tazobactam. Which of the following drugs is being described?
   Answer: Piperacillin
   Subject: Pharmacology

19. First line treatment of T315I mutation in CML management is?
   Answer: Ponatinib
   Subject: Medicine

51. A 23 year old lady diagnosed with Bell's palsy comes on the 3rd day after the symptom development. What would be the treatment of choice?
   Answer: Oral steroids + Acyclovir
   Subject: ENT

79. In a patient with culture-positive enterococcal endocarditis who has failed to respond to vancomycin because of resistance, the treatment most likely to be effective is :
   Answer: Linezolid
   Subject: Pharmacolog

In [7]:
# Save the merged data for future use
df.to_csv('medmcqa_with_categories.csv', index=False)
print("Saved merged data to medmcqa_with_categories.csv")


Saved merged data to medmcqa_with_categories.csv


In [8]:
# Additional analysis: Show category distribution by subject
print("Category distribution by medical subject:")
category_by_subject = pd.crosstab(df['subject_name'], df['llm_category'], margins=True)
category_by_subject


Category distribution by medical subject:


llm_category,Diagnosis,Diagnosis.,Medication,Other,Procedure,Trivia,All
subject_name,,,,,,,
Anaesthesia,7,0,2,0,8,0,17
Anatomy,15,0,0,0,2,0,17
Biochemistry,17,0,0,1,0,0,18
Dental,18,0,0,0,2,0,20
ENT,12,0,1,0,7,0,20
Forensic Medicine,13,0,1,1,2,0,17
Gynaecology & Obstetrics,15,0,1,0,4,0,20
Medicine,15,0,3,0,2,0,20
Microbiology,14,1,1,0,2,0,18


In [9]:
# Show all medication questions for detailed review
print(f"All {len(medication_questions)} medication questions:")
print("=" * 80)
for i, row in medication_questions.iterrows():
    print(f"\nQuestion {i+1}:")
    print(f"ID: {row['id']}")
    print(f"Subject: {row['subject_name']}")
    print(f"Topic: {row['topic_name']}")
    print(f"Question: {row['question']}")
    print(f"Correct Answer: {row['op' + 'abcd'[row['cop']]]}")
    print("-" * 40)


All 27 medication questions:

Question 8:
ID: 01dcc307-a222-4eb0-87b3-34dc9992b44b
Subject: Pharmacology
Topic: Cell Wall Synthesis Inhibitors
Question: This drug has activity against many strains of P. aeruginosa. However, when it is used alone, resistance has emerged during the course of treatment. The drug should not be used in penicillin-allergic patients. Its activity against gram-negative rods is enhanced if it is given in combination with tazobactam. Which of the following drugs is being described?
Correct Answer: Piperacillin
----------------------------------------

Question 19:
ID: 20442569-e3fd-409b-93f9-d92923cd6b15
Subject: Medicine
Topic: CLL & CML
Question: First line treatment of T315I mutation in CML management is?
Correct Answer: Ponatinib
----------------------------------------

Question 51:
ID: 24dea00f-a178-4c73-8dba-523a7415846b
Subject: ENT
Topic: nan
Question: A 23 year old lady diagnosed with Bell's palsy comes on the 3rd day after the symptom development. Wha